# W8 Result Evaluation

Evaluate baselines against pre-computed ground truth.

**Metric**: **Recall** = (number of baseline results with score ≤ GT's worst score) / (number of GT results).

i.e. if GT found n answers with worst score `an`, and baseline has k results scoring ≤ `an`, then recall = k/n.

In [1]:
import json
from pathlib import Path

RESULTS_DIR = Path(".")

# --- Choose query count: 10 or 100 ---
N_QUERIES = 100  # <-- change to 10 or 100

GT_FILE = f"w8_queries_{N_QUERIES}_gt.json"

def load_results(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def query_results_to_map(data):
    return {r["query_id"]: r for r in data["results"]}

# GT answer format: dict {id_t1, id_t2, score}
gt_data = load_results(RESULTS_DIR / GT_FILE)
gt_map = query_results_to_map(gt_data)

gt_pairs = {qid: set((row["id_t1"], row["id_t2"]) for row in r["answer"]) for qid, r in gt_map.items()}
gt_k = {qid: r.get("K", len(r["answer"])) for qid, r in gt_map.items()}

# Auto-discover baseline files (exclude GT), load once, sort by method name.
# Append run_id to the name so each row identifies its run.
_files = [f for f in RESULTS_DIR.glob("*.json")
          if f.name != GT_FILE and "_gt" not in f.name]
BASELINES = []  # list of (name, filename, data) sorted by name
for _f in _files:
    _d = load_results(_f)
    _base = _d.get("baseline", _d.get("method", _f.stem))
    _rid = _d.get("run_id")
    _name = f"{_base}@{_rid}" if _rid else _base
    BASELINES.append((_name, _f.name, _d))
BASELINES.sort(key=lambda x: x[0])

print(f"Ground truth: {GT_FILE}, queries = {len(gt_map)}")
print(f"Baselines ({len(BASELINES)}):")
for _name, _fname, _d in BASELINES:
    print(f"  {_name}  <-  {_fname}")

Ground truth: w8_queries_100_gt.json, queries = 100
Baselines (5):
  dase@20260415_005205  <-  20260415_005205.json
  dase@20260415_005300  <-  20260415_005300.json
  dase@20260415_005411  <-  20260415_005411.json
  dase@20260415_005509  <-  20260415_005509.json
  dase@20260415_005605  <-  20260415_005605.json


In [2]:
def eval_baseline(baseline_data, gt_map):
    """
    Recall = (# baseline results with score <= GT worst score) / (# GT results).
    Only evaluates on queries the baseline actually ran (so partial runs are
    not penalized by dividing by all GT queries).
    
    Baseline answer: [id_t1, id_t2, title_t1, title_t2, score]  (score at index -1)
    GT answer: dict {id_t1, id_t2, score}
    """
    bl_map = query_results_to_map(baseline_data)
    per_query = []
    
    for qid, bl_result in bl_map.items():
        gt_result = gt_map.get(qid)
        if gt_result is None:
            continue
        gt_answers = gt_result["answer"]
        n = len(gt_answers)
        k = gt_result.get("K", n)
        
        if n == 0:
            continue
        
        worst_gt_score = max(row["score"] for row in gt_answers)
        
        bl_answers = bl_result.get("answer", [])
        timed_out = bl_result.get("timed_out", False)
        
        hits = sum(1 for row in bl_answers if row[-1] <= worst_gt_score + 1e-6)
        recall = hits / n
        
        per_query.append({
            "query_id": qid, "K": k, "n_gt": n,
            "hits": hits, "recall": recall,
            "n_returned": len(bl_answers), "timed_out": timed_out,
        })
    
    total = len(per_query)
    mean_recall = sum(q["recall"] for q in per_query) / total if total else 0
    n_timeout = sum(1 for q in per_query if q["timed_out"])
    
    return per_query, mean_recall, n_timeout

In [3]:
summary = []
all_per_query = {}

for name, fname, data in BASELINES:
    per_query, mean_recall, n_timeout = eval_baseline(data, gt_map)

    total_sec = data.get("total_elapsed_sec", None)
    n_queries = len(per_query)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0

    summary.append({
        "baseline": name,
        "recall": mean_recall,
        "n_timeout": n_timeout,
        "n_queries": n_queries,
        "total_sec": round(total_sec, 1) if total_sec else None,
        "qps": qps,
    })
    all_per_query[name] = per_query

    print(f"{name}: recall={mean_recall:.4f}  qps={qps:.4f}  timeouts={n_timeout}")

dase@20260415_005205: recall=0.9795  qps=2.3425  timeouts=0
dase@20260415_005300: recall=0.9800  qps=2.1379  timeouts=0
dase@20260415_005411: recall=0.9800  qps=2.2714  timeouts=0
dase@20260415_005509: recall=0.9800  qps=2.2691  timeouts=0
dase@20260415_005605: recall=0.9800  qps=2.3341  timeouts=0


In [4]:
import pandas as pd

df = pd.DataFrame(summary)
print(df.to_string(index=False))

            baseline  recall  n_timeout  n_queries  total_sec      qps
dase@20260415_005205  0.9795          0        100       42.7 2.342544
dase@20260415_005300  0.9800          0        100       46.8 2.137914
dase@20260415_005411  0.9800          0        100       44.0 2.271371
dase@20260415_005509  0.9800          0        100       44.1 2.269056
dase@20260415_005605  0.9800          0        100       42.8 2.334115
